# TTS Client
To be ran in conjunction with `tts__engine`

In [1]:
import websockets
import openai
import asyncio

In [2]:
client = openai.OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [ ]:
response = client.chat.completions.create(
    model="cognitivecomputations_dolphin-2.9-llama3-8b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "tell me a short story (2 sentences)"
        }
    ],
    max_tokens=350,
    stream=True
)


import time
time_start = 0
# openai_gen = openai_generator(response)
# Establish a connection to the websocket server
websocket = await websockets.connect("ws://localhost:8000/ws")

async def feed_to_websocket(response):
    for chunk in response:
        print(chunk.choices[0].delta.content, end="")
        if chunk.choices[0].delta.content is None:
            break
        # Feed the chunk to the websocket
        await websocket.send(chunk.choices[0].delta.content)
         
    # Add a final END signal to the websocket
    await websocket.send("END")
    print("\n{:.2f}".format(time.time() - time_start), ":", "END sent")

async def receive_from_websocket(websocket):
    print("receive_from_websocket called")
    got_first_bunch = False
    try:
        counter = 0
        while True:
            # Message is either bytes or text "END"
            message = await websocket.recv()
            if message == "END":
                break
            if counter % 100 == 0:
                if not got_first_bunch:
                    print("{:.2f}".format(time.time() - time_start), ":", "First chunk received")
                    got_first_bunch = True
                # Print time every 100 messages
                print("{:.2f}".format(time.time() - time_start), ":", "100 chunks received")
            counter += 1

        print("Recv->", counter)
        print("RECEIVED END")
    except websockets.exceptions.ConnectionClosedError:
        print("Connection closed")
    except Exception as e:
        print(e)
        
async def gen():
    global time_start
    time_start = time.time()
    feeder = asyncio.create_task(feed_to_websocket(response))
    await receive_from_websocket(websocket)
        
    await asyncio.gather(feeder)
    print("Time taken:", time.time() - time_start)
    
    # Close the websocket connection
    await websocket.close()


await gen()

# engine_stream.stop()

# engine_stream.feed(gen()).play()

# 

receive_from_websocket called
Once upon a time, a curious little girl named Lucy discovered a secret garden. In the garden, hidden among the vibrant flowers, was a magical ancient tree. It sparkled like a star, and Lucy found her new special friend - an enchanting talking squirrel named Quiddle.None
0.46 : END sent
0.46 : First chunk received
1739650495.16 : 100 chunks received
1739650495.16 : 100 chunks received
1739650495.55 : 100 chunks received
1739650495.56 : 100 chunks received
1739650500.72 : 100 chunks received
1739650500.73 : 100 chunks received
1739650501.14 : 100 chunks received
1739650501.15 : 100 chunks received
1739650501.81 : 100 chunks received
1739650501.82 : 100 chunks received
1739650502.00 : 100 chunks received
1739650502.54 : 100 chunks received
1739650503.08 : 100 chunks received
1739650503.09 : 100 chunks received
1739650503.60 : 100 chunks received
1739650503.61 : 100 chunks received
1739650504.12 : 100 chunks received
1739650504.12 : 100 chunks received
1739650